In [ ]:
# root file producer for the 16x16
import os
import ROOT as root
import numpy as np
from array import array
import glob
import re
import pandas as pd
from openpyxl import Workbook
import pytz

file = root.TFile("/Users/icosivi/Desktop/PRE-SERIE/HPK/PRE-SERIE_HPK_Vendor_16x16_IV.root", "RECREATE")
tree = root.TTree("Tree","Tree")

event = array('i', [0])
wafer = array('i', [0])
sensor = array('i', [0])
row = array('i', [0])
column = array('i', [0])
temperature = array('f', [0])

V = root.std.vector("float")()
IBACK = root.std.vector("float")()

tree.Branch("wafer", wafer, 'wafer/I')
tree.Branch("event", event, 'event/I')
tree.Branch("sensor", sensor,'sensor/I')
tree.Branch("row", row, 'row/I')
tree.Branch("column", column, 'column/I')
tree.Branch("temperature", temperature, 'temperature/I')


V.reserve(1000)
IBACK.reserve(1000)
tree.Branch("V", "std::vector<float>", V)
tree.Branch("IBACK", "std::vector<float>", IBACK)

df_dict = pd.read_excel('/Users/icosivi/Desktop/PRE-SERIE/HPK/    .xlsx', sheet_name=None)

wb = pd.read_excel('/Users/icosivi/Desktop/PRE-SERIE/HPK/HPK_sensor-col-row.xlsx')

nevent = 0

for sheet_name, df in df_dict.items():
  wafer[0] = int(sheet_name)
  for i in range(24):
      V.clear()
      IBACK.clear()
      event[0] = nevent
      sensor[0] = int(df.iat[0,i+1])
      
      c, r = str(wb.loc[df['Sensor'] == int(i), ['Column', 'Row']].values[0])
      column[0] = int(c)
      row[0] = int(r)
      
      temperature[0] = int(df.iat[1,i+1])
      
      V_list = df[3:,0].dropna().tolist()
      for v in V_list:
        V.push_back( float(v) )
      
      I_list = df[3:,i+1].dropna().tolist()
      for i in I_list:
        IBACK.push_back( float(i) )
          
      
      nevent += 1
      
  

In [ ]:
# producer of the xls to register components on the database for the 16x16
xl_filename="HPK_16x16_PRE-SERIES"
ww = Workbook()
wws = wb.active
wws["A1"] = "SerialNumber"
wws["B1"] = "Vendor" 
wws["C1"] = "Batch" 
wws["D1"] = "Wafer" 
wws["E1"] = "Geometry"
wws["F1"] = "Row" 
wws["G1"] = "Column"
wws["H1"] = "Sensor Number"

for i in range(n_serialn):
    wws["A%i" %(serial_num_counter+1)] = serial_num
    wws["B%i" %(serial_num_counter+1)] = vendor
    wws["C%i" %(serial_num_counter+1)] = batch
    wws["D%i" %(serial_num_counter+1)] = Wafer
    wws["E%i" %(serial_num_counter+1)] = "16x16"
    
    Column, Row = str(wb.loc[df['Sensor'] == int(i), ['Column', 'Row']].values[0])
    
    wws["F%i" %(serial_num_counter+1)] = Row
    wws["G%i" %(serial_num_counter+1)] = Column
    wws["H%i" %(serial_num_counter+1)] = sensor_number

save_path = '/Users/icosivi/Desktop/PRE-SERIE/HPK/'
wb.save(save_path+xl_filename+".xlsx")